## **1. Setup**

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.data_pipeline import DatabaseManager, DataLoader, Preprocessor
from src.events import SimpleEventDetector

## **2. Load Data**

In [2]:
db_path = Path("../NordPoool/data/thesis_database.db")

db = DatabaseManager(db_path)
loader = DataLoader(db)
pre = Preprocessor()

df_prices = loader.load_prices()

## **3. Build DateTime**

In [3]:
import pandas as pd

df_prices["datetime"] = pd.to_datetime(df_prices["delivery_day"]) + pd.to_timedelta(df_prices["hour"], unit="h")

## **4. Detect Price Events**

In [4]:
event_detector = SimpleEventDetector()

df_price_events = event_detector.detect_price_events(df_prices)

## **5. Check Results**

In [5]:
df_price_events[[
    "datetime",
    "zone_id",
    "price_value",
    "price_delta",
    "abs_price_delta",
    "rolling_volatility_24h",
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility"
]].head()

,datetime,zone_id,price_value,price_delta,abs_price_delta,rolling_volatility_24h,low_price,high_price,price_spike,extreme_price,rapid_price_change,price_ramp_up,price_ramp_down,high_volatility
0,2020-01-01 00:00:00,1,28.78,NaN,NaN,NaN,False,False,False,False,False,False,False,False
20,2020-01-01 01:00:00,1,28.45,-0.33,0.33,NaN,False,False,False,False,False,False,False,False
40,2020-01-01 02:00:00,1,27.90,-0.55,0.55,NaN,False,False,False,False,False,False,False,False
60,2020-01-01 03:00:00,1,27.52,-0.38,0.38,NaN,False,False,False,False,False,False,False,False
80,2020-01-01 04:00:00,1,27.54,0.02,0.02,NaN,False,False,False,False,False,False,False,False


## **6. Count Events**

In [6]:
price_event_columns = [
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility"
]

event_counts = df_price_events[price_event_columns].sum().reset_index()

event_counts.columns = ["event_type", "count"]

event_counts

,event_type,count
0,low_price,100714
1,high_price,100790
2,price_spike,50385
3,extreme_price,10077
4,rapid_price_change,50393
5,price_ramp_up,50395
6,price_ramp_down,50389
7,high_volatility,100800


In [7]:
impact_summary = []

for event in price_event_columns:
    event_data = df_price_events[df_price_events[event]]
    non_event_data = df_price_events[~df_price_events[event]]

    impact_summary.append({
        "event_type": event,
        "event_count": event_data.shape[0],
        "avg_price_event": event_data["price_value"].mean(),
        "avg_price_non_event": non_event_data["price_value"].mean(),
        "median_price_event": event_data["price_value"].median(),
        "median_price_non_event": non_event_data["price_value"].median(),
        "avg_abs_delta_event": event_data["abs_price_delta"].mean(),
        "avg_abs_delta_non_event": non_event_data["abs_price_delta"].mean(),
        "avg_volatility_event": event_data["rolling_volatility_24h"].mean(),
        "avg_volatility_non_event": non_event_data["rolling_volatility_24h"].mean()
    })

impact_df = pd.DataFrame(impact_summary)
impact_df

,event_type,event_count,avg_price_event,avg_price_non_event,median_price_event,median_price_non_event,avg_abs_delta_event,avg_abs_delta_non_event,avg_volatility_event,avg_volatility_non_event
0,low_price,100714,2.093456,87.306477,1.58,60.08,3.756894,10.129279,17.890780,25.521430
1,high_price,100790,271.204372,57.415040,250.01,45.52,27.882189,7.449401,61.273754,20.701632
2,price_spike,50385,345.900386,64.738065,344.87,48.97,34.703902,8.166006,75.903660,22.067603
3,extreme_price,10077,514.096832,74.396494,524.77,51.52,48.259414,9.101076,105.380655,23.944711
4,rapid_price_change,50393,198.555844,72.489720,156.80,49.93,64.343655,6.605960,59.636323,22.923311
5,price_ramp_up,50395,186.782976,73.109030,145.03,49.86,48.810942,7.423300,48.693057,23.499207
6,price_ramp_down,50389,148.456742,75.126491,109.95,50.59,44.738611,7.637852,52.760144,23.285330
7,high_volatility,100800,210.505863,64.157110,172.60,47.68,27.389502,7.503922,77.279046,18.922584
